In [8]:
%pip install -q -U google-genai

In [9]:
from getpass import getpass
from google import genai

api_key = getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=api_key)

print("Done")

Paste your Gemini API key: ··········
Done


In [12]:
from google.colab import files

uploaded = files.upload()

pdf_paths = [
    name for name in uploaded.keys()
    if name.lower().endswith(".pdf")
]

print("PDFs uploaded:", pdf_paths)
assert len(pdf_paths) == 2, "Upload files"

Saving UNIT-III EPC updated.pdf to UNIT-III EPC updated (2).pdf
Saving UNIT-5 EPC.pdf to UNIT-5 EPC (2).pdf
PDFs uploaded: ['UNIT-III EPC updated (2).pdf', 'UNIT-5 EPC (2).pdf']


In [13]:
import time

store = client.file_search_stores.create(
    config={
        "display_name": "personaa-three-books",
        "embedding_model": "models/gemini-embedding-2"
    }
)

for path in pdf_paths:
    print("Indexing:", path)

    operation = client.file_search_stores.upload_to_file_search_store(
        file=path,
        file_search_store_name=store.name,
        config={"display_name": path}
    )

    while not operation.done:
        time.sleep(5)
        operation = client.operations.get(operation)

    print("Indexed:", path)

print("\nAll three books indexed!")

Indexing: UNIT-III EPC updated (2).pdf
Indexed: UNIT-III EPC updated (2).pdf
Indexing: UNIT-5 EPC (2).pdf
Indexed: UNIT-5 EPC (2).pdf

All three books indexed!


In [16]:
from google.genai import types

question = input("Ask a question about your books: ")

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=(
        "Answer using only information from the uploaded books. "
        "If the books do not contain the answer, say so. "
        "Cite the source when available.\n\n"
        + question
    ),
    config=types.GenerateContentConfig(
        tools=[
            types.Tool(
                file_search=types.FileSearch(
                    file_search_store_names=[store.name]
                )
            )
        ]
    )
)

print(response.text)

Ask a question about your books: What are the important topics
Based on the provided documents, the uploaded materials focus on **Sustainable Development** (UNIT-5). The important topics covered under this subject include:

* **Meaning and Definition:** Sustainable development as defined by the Brundtland Report (1987) as "development that meets present needs without compromising the ability of future generations to meet their own needs".
* **The Three Pillars of Sustainability:**
  * **Economic Sustainability:** Ensuring equitable economic growth that helps all parts of society without depleting natural resources or harming the environment.
  * **Environmental Sustainability:** Preserving the environment through the wise use of natural resources, reducing pollution, and mitigating climate change.
  * **Social Sustainability:** Advancing social justice, equity, inclusivity, and access to opportunities, fundamental necessities, and education.
* **Features and Principles of Sustainable D